# 06 — Detect + track on video, with a live health tally

Runs YOLO11n detection with **Ultralytics ByteTrack** persistent IDs over a video, overlays class-coloured boxes (green = healthy, orange-red = unhealthy) + track IDs + **live end-to-end FPS**, and writes an annotated output mp4.

Because IDs persist, the per-frame counts can be rolled up into a **per-plant verdict**: each track is scored by how many frames it was seen unhealthy, so one bad frame does not condemn a plant and one good frame does not clear it. That rollup is the number worth acting on — a single frame's count flickers with occlusion and blur.

Part 1 reassembles a LettuceMOTS sequence into an mp4 so you have a test clip. Part 2 runs the tracker on any video.

> Ships **unrun** — needs a trained checkpoint (03/04) and a source video. FPS here is measured around the **full** per-frame loop (read + inference + tracking + draw + write), not the model call alone.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

import cv2

# ===================== CONFIG (edit here only) =====================
WEIGHTS       = str(MODELS_DIR / "best.pt")
IMGSZ         = 640
CONF          = 0.25
IOU           = 0.50
TRACKER       = "bytetrack.yaml"    # Ultralytics ByteTrack config
DEVICE        = "cpu"               # 0 for GPU

# --- Part 1: reassemble a LettuceMOTS sequence into an mp4 for testing ---
LETTUCE_ROOT  = os.environ.get("LETTUCE_ROOT", r"D:\croprow_dataset\LettuceMOTS")
SEQ_SPLIT     = "test"              # "train" or "test"
SEQ_ID        = "0003"              # sequence folder to turn into a clip
REASSEMBLE_FPS = 10
REASSEMBLED_MP4 = str(RUNS_DIR / f"seq_{SEQ_SPLIT}_{SEQ_ID}.mp4")

# --- Part 2: tracking source + output ---
SOURCE_VIDEO  = REASSEMBLED_MP4     # or any mp4 path (e.g. your own footage)
OUTPUT_VIDEO  = str(RUNS_DIR / "tracked_health.mp4")
OUT_FPS       = REASSEMBLE_FPS

# A track is called unhealthy when at least this share of its sighted frames
# were unhealthy. 0.5 = majority vote over the plant's whole appearance.
TRACK_UNHEALTHY_FRAC = 0.5
# ===================================================================
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
print("source :", SOURCE_VIDEO)

## Part 1 — reassemble a sequence into mp4 (ffmpeg)

Frames are named `000000.png, 000001.png, ...`; ffmpeg stitches them at `REASSEMBLE_FPS` as H.264 + yuv420p (the codec browsers can actually play — `cv2`'s default `mp4v` cannot be decoded in a browser). Uncomment to run.

In [ ]:
# import subprocess
# try:
#     import imageio_ffmpeg
#     FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
# except ImportError:
#     FFMPEG = "ffmpeg"   # fall back to a system ffmpeg on PATH
#
# frames_dir = Path(LETTUCE_ROOT) / SEQ_SPLIT / "images" / SEQ_ID
# if not frames_dir.is_dir():
#     raise FileNotFoundError(f"sequence frames not found: {frames_dir}")
#
# first = sorted(frames_dir.glob("*.png"))[0]
# cmd = [
#     FFMPEG, "-y",
#     "-framerate", str(REASSEMBLE_FPS),
#     "-start_number", first.stem,
#     "-i", str(frames_dir / "%06d.png"),
#     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
#     REASSEMBLED_MP4,
# ]
# print(" ".join(cmd))
# subprocess.run(cmd, check=True)
# print("wrote", REASSEMBLED_MP4)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. See croprow_disease/requirements-train.txt.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install "
        "croprow_disease/requirements-train.txt into the croprow Python 3.11 venv "
        "before running this notebook."
    ) from e

## Part 2 — detect + ByteTrack + annotate

Manual OpenCV loop so the FPS timer wraps the entire pipeline. `persist=True` keeps track IDs stable across frames.

In [ ]:
import time
import collections
import numpy as np

if not Path(WEIGHTS).is_file():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}. Train first (03/04).")
if not Path(SOURCE_VIDEO).is_file():
    raise FileNotFoundError(f"Source video not found: {SOURCE_VIDEO}. Run Part 1 "
                            "or point SOURCE_VIDEO at an mp4.")

model = YOLO(WEIGHTS)
cap = cv2.VideoCapture(SOURCE_VIDEO)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), OUT_FPS, (W, H))

# track id -> [frames seen, frames seen unhealthy]
track_votes = collections.defaultdict(lambda: [0, 0])
n, fps_ema, t_all = 0, None, 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()

    res = model.track(frame, persist=True, tracker=TRACKER, imgsz=IMGSZ,
                      conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]

    n_healthy = n_unhealthy = 0
    boxes = res.boxes
    if boxes is not None and boxes.xyxy is not None and len(boxes):
        xyxy = boxes.xyxy.cpu().numpy().astype(int)
        clss = boxes.cls.cpu().numpy().astype(int)
        ids = (boxes.id.cpu().numpy().astype(int)
               if boxes.id is not None else np.full(len(xyxy), -1))
        for (x1, y1, x2, y2), cls, tid in zip(xyxy, clss, ids):
            color = U.CLASS_COLORS.get(int(cls), (200, 200, 200))
            if int(cls) == U.UNHEALTHY:
                n_unhealthy += 1
            else:
                n_healthy += 1
            if tid >= 0:
                track_votes[int(tid)][0] += 1
                track_votes[int(tid)][1] += int(cls == U.UNHEALTHY)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            tag = U.CLASS_NAMES[int(cls)]
            if tid >= 0:
                tag = f"#{tid} {tag}"
            cv2.putText(frame, tag, (x1, max(12, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    dt = time.perf_counter() - t0            # FULL loop: infer + track + draw
    t_all += dt
    inst = 1.0 / dt if dt > 0 else 0.0
    fps_ema = inst if fps_ema is None else 0.9 * fps_ema + 0.1 * inst

    cv2.putText(frame, f"FPS {fps_ema:5.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"healthy {n_healthy}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, U.CLASS_COLORS[U.HEALTHY], 2, cv2.LINE_AA)
    cv2.putText(frame, f"unhealthy {n_unhealthy}", (10, 88),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, U.CLASS_COLORS[U.UNHEALTHY], 2, cv2.LINE_AA)

    writer.write(frame)
    n += 1

cap.release()
writer.release()
mean_fps = n / t_all if t_all > 0 else 0.0
print(f"frames={n}  mean end-to-end FPS={mean_fps:.1f}")
print("wrote", OUTPUT_VIDEO)

## Per-plant rollup

One verdict per tracked plant instead of per frame. `unhealthy_frac` is the share of that plant's sighted frames in which it was called unhealthy; a plant is flagged when that clears `TRACK_UNHEALTHY_FRAC`.

Tracks seen in only a frame or two are listed but are weak evidence — a plant entering or leaving the frame edge gets few, poor looks.

In [ ]:
rows = []
for tid, (seen, bad) in sorted(track_votes.items()):
    frac = bad / seen if seen else 0.0
    rows.append((tid, seen, bad, frac, frac >= TRACK_UNHEALTHY_FRAC))

n_flagged = sum(1 for r in rows if r[4])
print(f"tracked plants: {len(rows)} | flagged unhealthy: {n_flagged}\n")
print(f"{'track':>6} {'frames':>7} {'unhealthy':>10} {'frac':>6}  verdict")
print("-" * 46)
for tid, seen, bad, frac, flag in rows:
    note = "  (few sightings)" if seen < 3 else ""
    print(f"{tid:>6} {seen:>7} {bad:>10} {frac:>6.2f}  "
          f"{'UNHEALTHY' if flag else 'healthy'}{note}")

Mean end-to-end FPS above is the honest number for this machine + config. See `07_speed` to sweep imgsz and exported backends against the 30 FPS target.